# 第１弾 STEP 1: Iceberg Table への書き込みと S3 物理構造の確認

## 目的
- Snowflake から Iceberg Table を作成・書き込みする
- S3 上の物理ファイル（metadata / manifest / data）の構造を確認する
- 書き込みのたびに metadata が更新されるスナップショットの仕組みを理解する

## 使用データ
Snowflake サンプルデータ（`SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS`）の一部を使用する

## STEP 0: 接続確認

In [ ]:
-- 接続・権限確認
SELECT CURRENT_USER(), CURRENT_ROLE(), CURRENT_WAREHOUSE(), CURRENT_DATABASE();

In [ ]:
-- External Volume の確認
SHOW EXTERNAL VOLUMES LIKE 'ICEBERG_S3_VOLUME';

In [ ]:
-- External Volume の詳細確認（S3 の storage_location が正しいか）
DESC EXTERNAL VOLUME ICEBERG_S3_VOLUME;

## STEP 1: 書き込み用 DB / Schema に切り替え

In [ ]:
USE DATABASE ICEBERG_DB;
USE SCHEMA WORK;
USE WAREHOUSE SANDBOX_WH;

## STEP 2: サンプルデータの確認

In [ ]:
-- TPCH サンプルデータの確認（件数・スキーマ）
SELECT COUNT(*) FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS;

In [ ]:
-- サンプル表示
SELECT * FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS LIMIT 10;

## STEP 3: Iceberg Table の作成

Snowflake managed Iceberg Table を作成する。  
`EXTERNAL_VOLUME` に作成済みの `ICEBERG_S3_VOLUME` を指定する。

In [ ]:
CREATE OR REPLACE ICEBERG TABLE ICEBERG_DB.WORK.ORDERS_ICEBERG (
    O_ORDERKEY      NUMBER(38, 0),
    O_CUSTKEY       NUMBER(38, 0),
    O_ORDERSTATUS   VARCHAR(1),
    O_TOTALPRICE    NUMBER(12, 2),
    O_ORDERDATE     DATE,
    O_ORDERPRIORITY VARCHAR(15),
    O_CLERK         VARCHAR(15),
    O_SHIPPRIORITY  NUMBER(38, 0),
    O_COMMENT       VARCHAR(79)
)
EXTERNAL_VOLUME = 'ICEBERG_S3_VOLUME'
CATALOG = 'SNOWFLAKE'
BASE_LOCATION = 'orders_iceberg/';

In [ ]:
-- テーブルが作成されたか確認
SHOW ICEBERG TABLES IN SCHEMA ICEBERG_DB.WORK;

## STEP 4: データの書き込み（1回目）

In [ ]:
-- 1回目: 1996年のデータを挿入
INSERT INTO ICEBERG_DB.WORK.ORDERS_ICEBERG
SELECT
    O_ORDERKEY, O_CUSTKEY, O_ORDERSTATUS, O_TOTALPRICE,
    O_ORDERDATE, O_ORDERPRIORITY, O_CLERK, O_SHIPPRIORITY, O_COMMENT
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS
WHERE YEAR(O_ORDERDATE) = 1996
LIMIT 10000;

In [ ]:
-- 件数確認
SELECT COUNT(*) FROM ICEBERG_DB.WORK.ORDERS_ICEBERG;

## STEP 5: S3 上の物理ファイル構造を確認

Snowflake の `SYSTEM$ICEBERG_TABLE_INFO()` 関数で metadata の場所を確認できる。

In [ ]:
-- Iceberg テーブルのメタデータ情報を確認
SELECT SYSTEM$ICEBERG_TABLE_INFO('ICEBERG_DB.WORK.ORDERS_ICEBERG');

In [ ]:
-- スナップショット一覧
SELECT *
FROM TABLE(
    INFORMATION_SCHEMA.ICEBERG_TABLE_SNAPSHOTS(
        TABLE_NAME => 'ORDERS_ICEBERG',
        TABLE_SCHEMA => 'WORK',
        TABLE_DATABASE => 'ICEBERG_DB'
    )
);

## STEP 6: データの書き込み（2回目）とスナップショットの変化

In [ ]:
-- 2回目: 1997年のデータを追加
INSERT INTO ICEBERG_DB.WORK.ORDERS_ICEBERG
SELECT
    O_ORDERKEY, O_CUSTKEY, O_ORDERSTATUS, O_TOTALPRICE,
    O_ORDERDATE, O_ORDERPRIORITY, O_CLERK, O_SHIPPRIORITY, O_COMMENT
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS
WHERE YEAR(O_ORDERDATE) = 1997
LIMIT 10000;

In [ ]:
-- スナップショットが増えたか確認（2件になるはず）
SELECT *
FROM TABLE(
    INFORMATION_SCHEMA.ICEBERG_TABLE_SNAPSHOTS(
        TABLE_NAME => 'ORDERS_ICEBERG',
        TABLE_SCHEMA => 'WORK',
        TABLE_DATABASE => 'ICEBERG_DB'
    )
)
ORDER BY COMMITTED_AT;

In [ ]:
-- 合計件数確認（20,000件になるはず）
SELECT COUNT(*) FROM ICEBERG_DB.WORK.ORDERS_ICEBERG;

## STEP 7: UPDATE / DELETE の挙動確認

Iceberg は更新・削除でも新しいスナップショットが作られる（COW: Copy-On-Write）。

In [ ]:
-- UPDATE: 特定ステータスの注文を更新
UPDATE ICEBERG_DB.WORK.ORDERS_ICEBERG
SET O_ORDERSTATUS = 'X'
WHERE O_ORDERSTATUS = 'F'
  AND YEAR(O_ORDERDATE) = 1996
  AND O_ORDERKEY < 100;

In [ ]:
-- スナップショットが3件になったか確認
SELECT SNAPSHOT_ID, COMMITTED_AT, OPERATION
FROM TABLE(
    INFORMATION_SCHEMA.ICEBERG_TABLE_SNAPSHOTS(
        TABLE_NAME => 'ORDERS_ICEBERG',
        TABLE_SCHEMA => 'WORK',
        TABLE_DATABASE => 'ICEBERG_DB'
    )
)
ORDER BY COMMITTED_AT;

## まとめ・気づき

- INSERT / UPDATE / DELETE のたびに新しいスナップショットが S3 の metadata/ 以下に追加される
- 実データは data/ 以下の Parquet ファイルとして格納される
- Snowflake Managed Iceberg は `CATALOG = 'SNOWFLAKE'` で Snowflake が catalog を管理する
- タイムトラベルや読み取り実験は `02_read_and_timetravel.ipynb` へ